# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [26]:
import os
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [27]:
MODEL_GPT = "gpt-4o-mini"
MODEL_LLAMA = "llama3.2"

load_dotenv(override=True)
openai = OpenAI(base_url="https://openrouter.ai/api/v1")
openai_audio = OpenAI()
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [28]:
SYSTEM_PROMPT = """You are a helpful technical tutor. Answer questions about code and software clearly.
Use markdown for code and structure. Be concise but thorough."""

In [30]:
def get_current_date():
    return datetime.now().strftime("%Y-%m-%d")

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_date",
            "description": "Return today's date in YYYY-MM-DD format.",
            "parameters": {"type": "object", "properties": {}, "additionalProperties": False},
        },
    }
]

In [31]:
def talker(message):
    try:
        response = openai_audio.audio.speech.create(
            model="gpt-4o-mini-tts",
            voice="onyx",
            input=message,
        )
        return response.content
    except Exception:
        return None

def transcribe(audio_path):
    if audio_path is None:
        return None
    if isinstance(audio_path, dict):
        audio_path = audio_path.get("name") or audio_path.get("path") or audio_path
    with open(audio_path, "rb") as f:
        out = openai_audio.audio.transcriptions.create(model="whisper-1", file=f)
    return out.text.strip() if out.text else None

In [32]:
def chat_for_blocks(history, model_name):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history

    if model_name == MODEL_GPT:
        response = openai.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools)
        while response.choices[0].finish_reason == "tool_calls":
            msg = response.choices[0].message
            messages.append(msg)
            for tc in msg.tool_calls:
                result = get_current_date() if tc.function.name == "get_current_date" else ""
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
            response = openai.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools)
        reply = response.choices[0].message.content or ""
    else:
        response = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages)
        reply = response.choices[0].message.content or ""

    history = history + [{"role": "assistant", "content": reply}]
    voice = talker(reply) if model_name == MODEL_GPT and reply else None
    return history, voice

In [33]:
def chat(message, history, model_name):
    history_messages = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history_messages + [{"role": "user", "content": message}]

    if model_name == MODEL_GPT:
        response = openai.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools)
        while response.choices[0].finish_reason == "tool_calls":
            msg = response.choices[0].message
            messages.append(msg)
            for tc in msg.tool_calls:
                result = get_current_date() if tc.function.name == "get_current_date" else ""
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
            response = openai.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools)
        text = response.choices[0].message.content or ""
        for i in range(1, len(text) + 1):
            yield text[:i]
        return

    stream = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        if chunk.choices:
            part = chunk.choices[0].delta.content or ""
            response += part
            yield response

In [34]:
model_selector = gr.Dropdown(choices=[MODEL_GPT, MODEL_LLAMA], value=MODEL_GPT, label="Model")
gr.ChatInterface(fn=chat, type="messages", additional_inputs=[model_selector], title="Technical Q&A").launch()

* Running on local URL:  http://127.0.0.1:7894
* To create a public link, set `share=True` in `launch()`.


In [35]:
def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]

with gr.Blocks(title="Technical Q&A (with audio)") as ui:
    chatbot = gr.Chatbot(height=500, type="messages")
    audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:", scale=4)
        model_selector = gr.Dropdown(choices=[MODEL_GPT, MODEL_LLAMA], value=MODEL_GPT, label="Model", scale=1)
    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat_for_blocks, inputs=[chatbot, model_selector], outputs=[chatbot, audio_output]
    )

ui.launch()

* Running on local URL:  http://127.0.0.1:7895
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/hakeemerisan/development/codes/andela/ai-engineering/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/hakeemerisan/development/codes/andela/ai-engineering/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/hakeemerisan/development/codes/andela/ai-engineering/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/hakeemerisan/development/codes/andela/ai-engineering/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1623, in call_function
    prediction =